### Libraries imports

In [111]:
import boto3
import pickle
import re
import os
import faiss
import numpy as np

from sentence_transformers import SentenceTransformer
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv(dotenv_path='../.env')

True

### Variables definition

In [112]:
# Fetch AWS credentials from the .env file
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION")

# Initialize the S3 client with the credentials
s3 = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_REGION
)

model = SentenceTransformer('all-MiniLM-L6-v2')

# Set your OpenAI API key
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
client = OpenAI()

/home/mikemadeira/anaconda3/envs/rag-aws-loka-env/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


### 1. Preprocessing the Text Files

In [56]:
def download_file_from_s3(s3, bucket_name, file_key, local_path='/tmp/downloaded_document.txt'):
    s3.download_file(bucket_name, file_key, local_path)
    print(f"Downloaded {file_key} from {bucket_name} to {local_path}")
    with open(local_path, 'r') as file:
        return file.read()

bucket_name = 'aws-test-docs-loka'
file_key = 'sagemaker_documentation/ml-creating-your-amazon-sagemaker-resource.md'
local_path = './ml-creating-your-amazon-sagemaker-resource.md'

text_file = download_file_from_s3(s3, bucket_name, file_key, local_path)


Downloaded sagemaker_documentation/ml-creating-your-amazon-sagemaker-resource.md from aws-test-docs-loka to ./ml-creating-your-amazon-sagemaker-resource.md


In [9]:
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def chunk_text(text, chunk_size=200, overlap=50):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = words[i:i + chunk_size]
        chunks.append(' '.join(chunk))
        i += chunk_size - overlap
    return chunks

chunks = chunk_text(clean_text(text_file))
print(f"Number of chunks: {len(chunks)}")

Number of chunks: 12


### 2. Generate embeddings

In [12]:
def generate_embeddings(chunks):
    embeddings = model.encode(chunks, convert_to_tensor=False)
    return embeddings

embeddings = generate_embeddings(chunks)
print(f"Generated {len(embeddings)} embeddings.")

Generated 12 embeddings.


### 3. Save Chunks and Embeddings to DynamoDB

#### Delete and create the DynamoDB table again

In [ ]:
# Define the variables
table_name = "aws_test_docs_chunks"
region = "eu-north-1"

# Navigate one directory up and make the script executable (optional)
!chmod +x ../create_dynamo_table.sh

# Run the script from one level above the current working directory
!source ../create_dynamo_table.sh {table_name} {region}

In [ ]:
# Initialize the DynamoDB resource and table
dynamodb = boto3.resource('dynamodb', region_name='eu-north-1')
table = dynamodb.Table('aws_test_docs_chunks')

In [77]:
# Function to save chunks and embeddings to DynamoDB with 'faiss_index' as a field
def save_chunks_and_embeddings_to_db(file_key, chunks, embeddings, faiss_index):
    """Save each chunk, its corresponding embedding, and FAISS index to DynamoDB."""
    for i, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
        unique_id = f'{file_key}_{i}'  # Create unique id by concatenating file_key and chunk index
        table.put_item(
            Item={
                'id': unique_id,  # Unique primary key
                'file_key': file_key,  # The original file the chunk came from
                'faiss_index': int(faiss_index),  # FAISS index
                'chunk': chunk,  # Text chunk
                'embedding': pickle.dumps(embedding)  # Serialized embedding
            }
        )
        faiss_index += 1  # Increment FAISS index
    print(f"Saved chunks and embeddings for {file_key} to DynamoDB")
    return faiss_index  # Return the updated FAISS index


In [ ]:
save_chunks_and_embeddings_to_db(file_key, chunks, embeddings)

In [71]:
# Check how the chunks are stored
response = table.scan()

print(f"Found {response['Count']} items")

all_embeddings = []

# Iterate through the items and print chunks and embeddings
for item in response['Items']:
    print(f"Item: {item}")
    chunk = item['chunk']
    file_key = item['file_key']
    faiss_index = item['faiss_index']
    # print(f"Chunk: {chunk}")
    print(f"File key: {file_key}")
    print(f"faiss_index: {faiss_index}")

Found 283 items
Item: {'faiss_index': Decimal('0'), 'embedding': Binary(b'\x80\x04\x95\x8c\x06\x00\x00\x00\x00\x00\x00\x8c\x15numpy.core.multiarray\x94\x8c\x0c_reconstruct\x94\x93\x94\x8c\x05numpy\x94\x8c\x07ndarray\x94\x93\x94K\x00\x85\x94C\x01b\x94\x87\x94R\x94(K\x01M\x80\x01\x85\x94h\x03\x8c\x05dtype\x94\x93\x94\x8c\x02f4\x94\x89\x88\x87\x94R\x94(K\x03\x8c\x01<\x94NNNJ\xff\xff\xff\xffJ\xff\xff\xff\xffK\x00t\x94b\x89B\x00\x06\x00\x00\x84s\x92\xbd\xc7^l\xbd\xfbI$<#\xcdE<deb=\xf8B;\xbd}\xf1^\xbdu\xe4-\xbc\xe66\xb1\xbc3\x9e\x96=`\x8c\x0e=\x8cL\xf3;\x92!\xdc\xbcG\x91H\xbc\xa4\x8bX=\xc2\xf1\xba=\xc2\xa1\xd1\xbb\xb7\xad\xe3\xbd\xad~\xa1\xbd\x0c\x97\x08\xbd7\xf8 \xbd\xdf\xd0\xb5\xbb\xda\x126\xbdP\xa2\xa6<\x0c\xe9@;\xc3\x98\x83\xbdk\xcc\xc2\xbd\xb4\xe4\x07<\x93\x8a\x83<[3\xf7\xbcF\xf4\xe8\xbb\x8aI2\xbc\n\xf7J;\xf4X;:\xef{\xb6;e\xa8\x02>\x7f\xb2\n:\x19S\x82\xbd\xaf\xbeH\xbc=\xa6T=\xd2\x0c\xac=c&\xc1\xbc;\xbb\xe4\xbd\x10`0\xbd\xa82\xb3<\x9d\xeb@<m\xa2\xf8<QV6\xbd\xcaI\x91=\xad\x87J\xbb\xa3\xe5

### 4. Create and save FAISS index in S3 bucket

In [23]:
def create_faiss_index(embeddings):
    embeddings_np = np.array(embeddings)
    dimension = embeddings_np.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings_np)
    return index

def save_faiss_index_to_s3(faiss_index, s3_bucket, s3_key):
    """Save the FAISS index to S3."""
    local_index_path = './faiss_index.index'  # Save in /tmp directory
    faiss.write_index(faiss_index, local_index_path)  # Save to /tmp directory first
    s3.upload_file(local_index_path, s3_bucket, s3_key)
    print(f"FAISS index saved to S3 at {s3_bucket}/{s3_key}.")

faiss_index = create_faiss_index(embeddings)

save_faiss_index_to_s3(faiss_index, bucket_name, 'faiss_index.index')

FAISS index saved to aws-test-docs-loka/faiss_index.index


### 5. Query the FAISS index

In [24]:
# Load FAISS index from S3
def load_faiss_index_from_s3(s3_bucket, s3_key, local_index_file='faiss_index.index'):
    """Download FAISS index from S3 and load it locally."""
    s3.download_file(s3_bucket, s3_key, local_index_file)
    faiss_index = faiss.read_index(local_index_file)
    return faiss_index

In [100]:
def retrieve_chunks_from_db(query_embedding, faiss_index, top_k=5):
    """Retrieve chunks from DynamoDB using the FAISS indices."""
    
    distances, indices = faiss_index.search(query_embedding, top_k)
    print(f"Distances: {distances}")
    print(f"Indices: {indices}")

    results = []
    source_files = []

    for idx in indices[0]:  # Loop through the top-k indices returned by FAISS
        try:
            """Retrieve chunks from DynamoDB using faiss_index."""

            # Convert FAISS index from numpy.int64 to Python int
            idx = int(idx)
            
            response = table.query(
                IndexName='faiss_index_gsi',  # Use the GSI for faiss_index
                KeyConditionExpression=boto3.dynamodb.conditions.Key('faiss_index').eq(idx)
            )
            print(f"Table item: {response}")
            # Check if any items were returned
            if 'Items' in response and response['Items']:
                print(f"Response Item: {response['Items'][0]}")
                chunk = response['Items'][0]['chunk']
                file_key = response['Items'][0]['file_key']  # Retrieve the associated file key
                
                # Append the chunk and file key to the results
                results.append(chunk)
                source_files.append(file_key)
                
                print(f"Chunk: {chunk}")
                print(f"File key: {file_key}")
            else:
                print(f"No item found for FAISS index: {idx}")
        except Exception as e:
            print(f"Error retrieving chunk for index {idx}: {str(e)}")
    
    return results, source_files

In [32]:
query = "Why Should You Use MLOps?"
query_embedding = model.encode([query])

# Load FAISS index from S3
s3_index_bucket = 'aws-test-docs-loka'
s3_index_key = 'faiss_index.index'
faiss_index = load_faiss_index_from_s3(s3_index_bucket, s3_index_key)

retrieved_chunks = retrieve_chunks_from_db(query_embedding, faiss_index)

Distances: [[1.6356009 1.6394238 1.7467489 1.7520483 1.7986364]]
Indices: [[11  8  5  0 10]]
Table item: {'Item': {'faiss_index': Decimal('11'), 'embedding': Binary(b'\x80\x04\x95\x8c\x06\x00\x00\x00\x00\x00\x00\x8c\x15numpy.core.multiarray\x94\x8c\x0c_reconstruct\x94\x93\x94\x8c\x05numpy\x94\x8c\x07ndarray\x94\x93\x94K\x00\x85\x94C\x01b\x94\x87\x94R\x94(K\x01M\x80\x01\x85\x94h\x03\x8c\x05dtype\x94\x93\x94\x8c\x02f4\x94\x89\x88\x87\x94R\x94(K\x03\x8c\x01<\x94NNNJ\xff\xff\xff\xffJ\xff\xff\xff\xffK\x00t\x94b\x89B\x00\x06\x00\x00\x05\xc1\x83\xbd\xca.\xba\xbc\rle\xbdV\x7f*\xbc\xc3\x1d\xdb=\x10\xbf,\xbd\xe4m\xaa\xbd\xb8\xe2\x8b\xbd\xde\xff\r\xbd84q\xbcJ\xa2\xe6<-J\x82\xbc\xdb\xcf\x83=\x1e\x1bn\xbd\xe5r\xe4<\xce D\xbd#\x93\xf6<\x96\xf1\xf9\xbc\xcd\x8a\xd3\xbd\xe0\x06\xc4\xbd\xb9\xd5\x07\xbd\xa1>\x05<%\xc9\x91\xbdl\xd0\x96=\x1d\xcc\x19\xbdR\xe1\x97=\xc7\xbbf\xbd\xb8&q\xbd\xfe\x8c]=:g\xb7\xbd\xac\xa7\xa5=\xdf\xb7\xd1\xbc=\xb4\x8f<\xed,\x9d\xba\xc5\xe7\x11>\xf9ml=\x0c]\xd4\xbcPk\xb3\xbd\xa6(\x9

In [ ]:
retrieved_chunks

### 6. Process all files in S3 bucket

### Create S3 bucket and upload documents to test the pipeline

In [73]:
# Define the variables
script_path = "../create_s3_bucket.sh"
bucket_name = "faiss-test-docs-loka"
region = "eu-north-1"

# Navigate one directory up and make the script executable (optional)
!chmod +x {script_path}

!source {script_path} {bucket_name} {region}

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


{
    "Location": "http://faiss-test-docs-loka.s3.amazonaws.com/"
}
S3 bucket faiss-test-docs-loka created in region eu-north-1.


In [104]:
# Define the variables
script_path = "../upload_to_s3.py"
source_folder = "../test_set/"
bucket_name = "faiss-test-docs-loka"
folder_name = "sagemaker_documentation"

# Run the python script with the variables
!python {script_path} {source_folder} {bucket_name} {folder_name}

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


File ../test_set/aws-properties-sagemaker-dataqualityjobdefinition-dataqualityappspecification.md uploaded successfully to faiss-test-docs-loka/sagemaker_documentation/aws-properties-sagemaker-dataqualityjobdefinition-dataqualityappspecification.md.
File ../test_set/aws-properties-sagemaker-modelbiasjobdefinition-networkconfig.md uploaded successfully to faiss-test-docs-loka/sagemaker_documentation/aws-properties-sagemaker-modelbiasjobdefinition-networkconfig.md.
File ../test_set/aws-properties-sagemaker-dataqualityjobdefinition-batchtransforminput.md uploaded successfully to faiss-test-docs-loka/sagemaker_documentation/aws-properties-sagemaker-dataqualityjobdefinition-batchtransforminput.md.
File ../test_set/aws-properties-sagemaker-workteam-notificationconfiguration.md uploaded successfully to faiss-test-docs-loka/sagemaker_documentation/aws-properties-sagemaker-workteam-notificationconfiguration.md.
File ../test_set/aws-properties-sagemaker-modelpackage-bias.md uploaded successfully

#### Delete and create the DynamoDB table again

In [105]:
# Define the variables
table_name = "faiss_test_docs_chunks"
region = "eu-north-1"

# Navigate one directory up and make the script executable (optional)
!chmod +x ../create_dynamo_table.sh

# Run the script from one level above the current working directory
!source ../create_dynamo_table.sh {table_name} {region}

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


{
    "TableDescription": {
        "TableName": "faiss_test_docs_chunks",
        "TableStatus": "DELETING",
        "ProvisionedThroughput": {
            "NumberOfDecreasesToday": 0,
            "ReadCapacityUnits": 0,
            "WriteCapacityUnits": 0
        },
        "TableSizeBytes": 0,
        "ItemCount": 0,
        "TableArn": "arn:aws:dynamodb:eu-north-1:872515285249:table/faiss_test_docs_chunks",
        "TableId": "39b763d6-78f3-4ca8-9113-0de84af4f9ed",
        "BillingModeSummary": {
            "BillingMode": "PAY_PER_REQUEST",
            "LastUpdateToPayPerRequestDateTime": 1727631102.297
        },
        "DeletionProtectionEnabled": false
    }
}
DynamoDB table faiss_test_docs_chunks deleted.
{
    "TableDescription": {
        "AttributeDefinitions": [
            {
                "AttributeName": "faiss_index",
                "AttributeType": "N"
            },
            {
                "AttributeName": "id",
                "AttributeType": "S"
         

### Define variables

In [106]:
# Define the variables
s3_index_bucket = 'faiss-test-docs-loka'
s3_index_key = 'faiss_index.index'

table_name = "faiss_test_docs_chunks"
region = "eu-north-1"

# Initialize the DynamoDB resource and table
dynamodb = boto3.resource('dynamodb', region_name=region)
table = dynamodb.Table(table_name)

In [107]:
def process_files(bucket_name, folder_name, s3_index_bucket, s3_index_key):
    response = s3.list_objects_v2(Bucket=bucket_name, Prefix=folder_name)
    
    all_embeddings = []
    faiss_index = 0  # Initialize the FAISS index counter

    for obj in response.get('Contents', []):
        file_key = obj['Key']  # Get the file path in S3
        print(f"Processing file: {file_key}")

        # Step 1: Download file from S3
        text = download_file_from_s3(s3, bucket_name, file_key)
        
        # Step 2: Clean and chunk the text
        chunks = chunk_text(text)
        print(f"File: {file_key} - Number of chunks: {len(chunks)}")
        
        # Step 3: Generate embeddings
        embeddings = generate_embeddings(chunks)
        
        # Step 4: Save chunks and embeddings to DynamoDB and keep track of FAISS indices
        faiss_index = save_chunks_and_embeddings_to_db(file_key, chunks, embeddings, faiss_index)
        
        all_embeddings.append(embeddings)

    # Combine all embeddings and create FAISS index
    if all_embeddings:
        combined_embeddings = np.vstack(all_embeddings)
        print(f"Combined embeddings shape: {combined_embeddings.shape}")
        faiss_index_obj = create_faiss_index(combined_embeddings)
        save_faiss_index_to_s3(faiss_index_obj, s3_index_bucket, s3_index_key)
    else:
        print("No valid embeddings found.")


In [108]:
folder_name = 'sagemaker_documentation/'

process_files(bucket_name, folder_name, s3_index_bucket, s3_index_key)

Processing file: sagemaker_documentation/amazon-sagemaker-toolkits.md
Downloaded sagemaker_documentation/amazon-sagemaker-toolkits.md from faiss-test-docs-loka to /tmp/downloaded_document.txt
File: sagemaker_documentation/amazon-sagemaker-toolkits.md - Number of chunks: 4
Saved chunks and embeddings for sagemaker_documentation/amazon-sagemaker-toolkits.md to DynamoDB
Processing file: sagemaker_documentation/asff-resourcedetails-awssagemaker.md
Downloaded sagemaker_documentation/asff-resourcedetails-awssagemaker.md from faiss-test-docs-loka to /tmp/downloaded_document.txt
File: sagemaker_documentation/asff-resourcedetails-awssagemaker.md - Number of chunks: 1
Saved chunks and embeddings for sagemaker_documentation/asff-resourcedetails-awssagemaker.md to DynamoDB
Processing file: sagemaker_documentation/automating-sagemaker-with-eventbridge.md
Downloaded sagemaker_documentation/automating-sagemaker-with-eventbridge.md from faiss-test-docs-loka to /tmp/downloaded_document.txt
File: sagema

In [109]:
query = "Why Should You Use MLOps?"
query_embedding = model.encode([query])

# Load FAISS index from S3
faiss_index = load_faiss_index_from_s3(s3_index_bucket, s3_index_key)

retrieved_chunks = retrieve_chunks_from_db(query_embedding, faiss_index)

Distances: [[0.8726399 0.880074  0.8870468 1.0624946 1.0845742]]
Indices: [[849 844 845 847 848]]
Table item: {'Items': [{'faiss_index': Decimal('849'), 'embedding': Binary(b'\x80\x04\x95\x8c\x06\x00\x00\x00\x00\x00\x00\x8c\x15numpy.core.multiarray\x94\x8c\x0c_reconstruct\x94\x93\x94\x8c\x05numpy\x94\x8c\x07ndarray\x94\x93\x94K\x00\x85\x94C\x01b\x94\x87\x94R\x94(K\x01M\x80\x01\x85\x94h\x03\x8c\x05dtype\x94\x93\x94\x8c\x02f4\x94\x89\x88\x87\x94R\x94(K\x03\x8c\x01<\x94NNNJ\xff\xff\xff\xffJ\xff\xff\xff\xffK\x00t\x94b\x89B\x00\x06\x00\x00\x0e\x05X\xbdH\x0e`\xbdw\xfb\xd1<\xf9\x19\x81=r\xe5D=\x84\x9d\x92\xbb*\xb1\xb0\xbd\x8b\xfcS\xbd\xec\x9e\xc8\xbb\x18\t\xc9<-L\xf9<_\x053=\xa2<\x91=(\xa5\xb9\xbdH\tF\xbd\xea^`\xbc\xe5V\xbd=dB\xfc8\x8b\xf4\x85\xbd\xaf\xe3\x01\xbdp\xea*\xbdE\xdf=;\xd2\xe8\xc0\xbdOt\xa8=r\xa5\xdd\xbc\xa7\xd8\xb4<Q1\x9f\xbc\x1c\xbd\x81\xbc\rP\x0b=\xed\xf2\x90\xbb\\\x164\xbd\xae\xc0\x18\xbc\xdb\x9cK<}\t7=\x03\xb2\x13\xbd\xe4\xed\x06<.\x08V=\x8e@\xa7\xbc\x8a(\x9b;s\xbe\xb8\xbci\x9

In [110]:
retrieved_chunks

(['and waste less time with missing or invalid data\\. + **Repeatability**: Automating all the steps in the MLDC helps you ensure a repeatable process, including how the model is trained, evaluated, versioned, and deployed\\. + **Reliability**: Incorporating CI/CD practices allows for the ability to not only deploy quickly but with increased quality and consistency\\. + **Auditability**: Versioning all inputs and outputs, from data science experiments to source data to trained model, means that we can demonstrate exactly how the model was built and where it was deployed\\. + **Data and model quality**: MLOps lets us enforce policies that guard against model bias and track changes to data statistical properties and model quality over time\\.',
  '# Why Should You Use MLOps?<a name="sagemaker-projects-why"></a> As you move from running individual artificial intelligence and machine learning \\(AI/ML\\) projects to using AI/ML to transform your business at scale, the discipline of ML Oper

In [117]:
# Generate Answer using LLM
def generate_answer(query, retrieved_chunks):
    """Generate an answer using an LLM based on the query and retrieved chunks."""
    context = "\n".join(retrieved_chunks)  # Combine all chunks into a single context
    
    # Create the prompt for the LLM
    prompt = f"Answer the following question based on the provided context:\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"
    
    # Send the prompt to the LLM (using OpenAI's GPT)
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{"role": 'system', "content": 'You are a helpful assistant.'}, 
                  {"role": 'user', "content": prompt}],
    )
    
    print(f"Response from the LLM:\n{response}")
    # Extract the answer from the response
    answer = response.choices[0].message.content
    
    return answer

In [118]:
# Generate answer
answer = generate_answer(query, retrieved_chunks[0])
print("Answer:", answer)
print("For further reading:", retrieved_chunks[1])

Response from the LLM:
ChatCompletion(id='chatcmpl-ACwXxAPbnQ6KK2JL5lkp3usjFzL8w', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='You should use MLOps because it enhances the efficiency and effectiveness of your machine learning projects. By integrating MLOps practices, you can:\n\n1. **Improve Delivery Time**: Streamlining the machine learning development lifecycle helps accelerate the deployment of ML models.\n2. **Reduce Defects**: Implementing quality assurance measures ensures higher quality models with fewer errors.\n3. **Increase Productivity**: Offering self-service environments with access to curated datasets enables data engineers and scientists to work more efficiently and minimizes time lost due to data issues.\n4. **Ensure Repeatability**: Automating the steps in the machine learning development cycle promotes a consistent and repeatable process for model training, evaluation, versioning, and deployment.\n5. **Enhance R

In [103]:
# Check how the chunks are stored
response = table.scan()

print(f"Found {response['Count']} items")

all_embeddings = []

# Iterate through the items and print chunks and embeddings
for item in response['Items']:
    # print(f"Item: {item}")
    chunk = item['chunk']
    file_key = item['file_key']
    faiss_index = item['faiss_index']
    # print(f"Chunk: {chunk}")
    print(f"File key: {file_key}")
    print(f"faiss_index: {faiss_index}")

Found 62 items
File key: sagemaker_documentation/kubernetes-sagemaker-jobs.md
faiss_index: 20
File key: sagemaker_documentation/kubernetes-sagemaker-jobs.md
faiss_index: 11
File key: sagemaker_documentation/kubernetes-sagemaker-jobs.md
faiss_index: 4
File key: sagemaker_documentation/kubernetes-sagemaker-jobs.md
faiss_index: 33
File key: sagemaker_documentation/kubernetes-sagemaker-jobs.md
faiss_index: 21
File key: sagemaker_documentation/sagemaker-projects-templates-sm.md
faiss_index: 58
File key: sagemaker_documentation/kubernetes-sagemaker-jobs.md
faiss_index: 17
File key: sagemaker_documentation/kubernetes-sagemaker-jobs.md
faiss_index: 38
File key: sagemaker_documentation/kubernetes-sagemaker-jobs.md
faiss_index: 24
File key: sagemaker_documentation/sagemaker-projects-templates-sm.md
faiss_index: 49
File key: sagemaker_documentation/kubernetes-sagemaker-jobs.md
faiss_index: 31
File key: sagemaker_documentation/sagemaker-projects-templates-sm.md
faiss_index: 56
File key: sagemaker_